# Projected VQE — variation *after* projection

Ask for the lowest state *with one particle* and an ordinary VQE will not
answer the question: a hardware-efficient ansatz does not conserve particle
number, so the optimizer is free to leave the sector entirely and report the
energy of a state you never asked about.

`ProjectedVQE` closes that door by optimizing the **projected Rayleigh
quotient**

$$E_P(\boldsymbol\theta) =
  \frac{\langle \psi(\boldsymbol\theta)|P^\dagger H P|\psi(\boldsymbol\theta)\rangle}
       {\langle \psi(\boldsymbol\theta)|P^\dagger P|\psi(\boldsymbol\theta)\rangle}$$

for a symmetry projector `P`. Every value it reports is an energy *of the
sector*.

This is the complement of `examples/blocks/mwe_projected_symmetry_states.ipynb`,
which projects **after** the variation — it runs a VQE to convergence, then
applies the projector block once to clean up the answer. Projecting inside the
objective instead changes what is being minimized, and the two land on
different states.

In [ ]:
import numpy as np

import qarpx as qx
from qarp.algorithms import ProjectedVQE, VQE
from qarp.blocks import HEABlock, ParticleNumberProjectorBlock
from qarp.errors import CapabilityError
from qarp.optimizers import ScipyOptimizer
from qarp.utils import FH_ham_and_wf_singles_and_doubles

N_QUBITS = 4
N_PARTICLES = 1
T_HOP = 1.4

# 2-site Fermi-Hubbard under Jordan-Wigner: 2 sites x 2 spins = 4 qubits.
hamiltonian, _ = FH_ham_and_wf_singles_and_doubles(2, t=T_HOP, generalised=False)

# Three layers: a shallower HEA cannot express this ground state, and a VQE
# that cannot reach the minimum would not demonstrate anything about sectors.
ansatz = HEABlock(
    n_qubits=N_QUBITS, n_layers=3, real=False, linear=True, circular=True, use_cz=False
)
ansatz.build()

start = np.linspace(0.1, 1.0, len(ansatz.symbols))
print("HEA parameters:", len(ansatz.symbols))

## The oracle

Two independent checks, neither of them another qarp run.

**Analytic.** One particle hopping on two sites is the textbook two-level
problem: the bonding and antibonding combinations sit at `∓t`. So the exact
ground energy of the `N = 1` sector is `−t = −1.4`, with no interaction term
contributing (a single particle has nothing to interact with).

**Dense.** The same number from `eigvalsh` of the Hamiltonian matrix
restricted to the sector. qarp is LSB throughout — `op.sparse_matrix()`
included — so basis index `s` has qubit `q` occupied exactly when bit `q` of
`s` is set, which makes `s.bit_count()` the particle number of that basis
state.

In [ ]:
matrix = hamiltonian.sparse_matrix(N_QUBITS).toarray()

sector = [s for s in range(2**N_QUBITS) if s.bit_count() == N_PARTICLES]
outside = [s for s in range(2**N_QUBITS) if s not in sector]

global_ground = np.linalg.eigvalsh(matrix)[0]
sector_ground = np.linalg.eigvalsh(matrix[np.ix_(sector, sector)])[0]

print(f"sector dimension        : {len(sector)} of {2**N_QUBITS}")
print(f"N=1 ground (dense)      : {sector_ground:.10f}")
print(f"N=1 ground (analytic -t): {-T_HOP:.10f}")
print(f"global ground energy    : {global_ground:.10f}")

assert np.isclose(sector_ground, -T_HOP, atol=1e-12)

Note the ordering that makes this example work: the global ground energy is
**below** the sector floor. The lowest state overall has two particles, not
one, so "minimize `⟨H⟩`" and "minimize `⟨H⟩` within `N = 1`" have genuinely
different answers, and an unconstrained optimizer will always prefer the
former.

In [ ]:
assert global_ground < sector_ground
print(f"{global_ground:.6f} (global, N=2)  <  {sector_ground:.6f} (N=1 floor)")

## Without the projector

A plain VQE over the same ansatz minimizes over the whole Hilbert space. It
converges — to the global ground state, which has essentially **zero** weight
in the sector we asked about. The number it reports is below the `N = 1`
floor, which is the tell: it is not an `N = 1` energy at all.

In [ ]:
plain = VQE(
    operator=hamiltonian,
    ket=ansatz,
    initial_parameters=start,
    gradient=True,
    optimizer=ScipyOptimizer("L-BFGS-B", options={"maxiter": 2000}),
)
plain.build()
plain.suppress_success_message = True
plain_energy, plain_params = plain.run()

# parameter_map() keeps the canonical symbol order; never hand-zip against it.
bound = ansatz.set_symbols(ansatz.parameter_map(plain_params))
plain_state = np.asarray(qx.QarpSimulator().statevector(bound.flatten(), bound.n_qubits))
plain_weight = float(sum(abs(plain_state[s]) ** 2 for s in sector))

print(f"plain VQE energy      : {plain_energy:.10f}")
print(f"global ground         : {global_ground:.10f}")
print(f"weight inside N=1     : {plain_weight:.3e}")
print(f"below the N=1 floor   : {plain_energy < sector_ground}")

assert np.isclose(plain_energy, global_ground, atol=1e-6)
assert plain_weight < 1e-6

## With the projector

`ProjectedVQE` takes the same ansatz and a projector — here
`ParticleNumberProjectorBlock`. Pass a block (or a sequence of blocks, whose
sectors intersect) and it builds the dense projector for you; pass
`projector_matrix=` to supply one directly.

In [ ]:
pvqe = ProjectedVQE(
    operator=hamiltonian,
    ket=ansatz,
    projector=ParticleNumberProjectorBlock(n_qubits=N_QUBITS, Npart=N_PARTICLES),
    initial_parameters=start,
    optimizer=ScipyOptimizer("COBYLA", options={"maxiter": 3000}),
)
pvqe.build()
pvqe.suppress_success_message = True

energy, parameters = pvqe.run()

print(f"ProjectedVQE energy   : {energy:.10f}")
print(f"analytic -t           : {-T_HOP:.10f}")
print(f"error                 : {abs(energy + T_HOP):.2e}")

assert np.isclose(energy, -T_HOP, atol=1e-6)

### The variational bound

The objective is a Rayleigh quotient over the projected subspace, so the
converged value is bounded below by the exact ground energy of `H` **restricted
to the sector** — and *not* by the global ground energy, which lies below it.
That one-sided bound is the property worth testing.

In [ ]:
assert energy >= sector_ground - 1e-9, (energy, sector_ground)

print(f"E_P        = {energy:.10f}")
print(f"E_sector   = {sector_ground:.10f}   <- E_P is bounded below by this")
print(f"E_global   = {global_ground:.10f}   <- and is NOT bounded below by this")

### The state really is in the sector

`final_ansatz_statevector` is the raw circuit output, which still leaks across
sectors — the projector did not make the ansatz symmetry-preserving.
`final_projected_statevector` is `P|ψ⟩` renormalized: the state the reported
energy actually belongs to.

In [ ]:
raw = pvqe.final_ansatz_statevector
projected = pvqe.final_projected_statevector

print(f"ansatz weight inside N=1    : {sum(abs(raw[s]) ** 2 for s in sector):.6f}")
print(f"projected max |amp| outside : {max(abs(projected[s]) for s in outside):.3e}")
print(f"projected norm              : {np.linalg.norm(projected):.12f}")

assert max(abs(projected[s]) for s in outside) < 1e-10
assert np.isclose(np.linalg.norm(projected), 1.0, atol=1e-10)

In [ ]:
# Particle-number weight distribution, before and after the projector.
for label, state in (("ansatz", raw), ("projected", projected)):
    weights = np.zeros(N_QUBITS + 1)
    for s in range(2**N_QUBITS):
        weights[s.bit_count()] += abs(state[s]) ** 2
    print(f"{label:10s}", "  ".join(f"N={n}:{w:6.4f}" for n, w in enumerate(weights)))

## Why the objective refuses `"parameter-shift"`

The projected quotient is a **ratio** of two quadratic forms, not a bilinear
expectation value, so the shift rules have nothing to stand on: the value is
not a trigonometric polynomial in the gate angles. The primitive declares
`gradient_kind = "none"` and the analytic path is refused rather than returned
wrong. `"finite-diff"` still applies.

This is the `"none"` row of the registry in
`examples/engines/mwe_gradients.ipynb`.

In [ ]:
print("gradient_kind:", pvqe.primitive.gradient_kind)

point = ansatz.parameter_map(parameters)

try:
    pvqe.engine.run_gradient(point, method="parameter-shift")
except CapabilityError as exc:
    print("\nparameter-shift ->", exc)

fd = pvqe.engine.run_gradient(point, method="finite-diff", options={"fd_eps": 1e-6})[0]
print(f"\nfinite-diff      -> ok, |grad| = {np.linalg.norm(fd):.3e} at the converged point")

## A note on cost

`ProjectedVQE` does **not** simulate the projector circuit on every objective
call. The engine compiles the bare ansatz once; each evaluation substitutes
parameters, simulates that circuit, and applies the dense projector
algebraically. The block-encoded projector stays useful for circuit-level
demonstrations and hardware-style postselection — it is simply far too
expensive as an inner optimizer primitive.

That is why the projector arrives as a `Block` for convenience but is consumed
as a matrix: `projector_matrix=` bypasses the block entirely when you already
have one, and is the route to a projector with no circuit representation.